# Workshop 4: Hands-on — Building Cooperative LLM Agent Workflows for Anti-pattern detection, Code Smell and Technical Debt Resolution
**LLMA4SE 2026** · 3 hours · CPU · OpenRouter + official `openai` client.

Key takeaway: **tools measure · the LLM interprets · a gate decides.**

**For instance.** A reviewer pastes a billing function into a chat window and asks
“What is this complexity?” The model says “cyclomatic complexity 4.” **radon** says **7**. If you had
gated a merge on the chat reply, you would have shipped a lie. Today we build a *team* that
cannot certify itself:

- **[PyExamine](https://github.com/KarthikShivasankar/python_smells_detector)** (`code-quality-analyzer`, MSR 2025) measures *code / structural / architectural* smells.
- **[MLScent](https://github.com/KarthikShivasankar/ml_smells_detector)** (`ml-code-smell-detector`, CAIN 2025) measures *ML anti-patterns* (leakage, seeds, eval mode).

- [**Radon**](https://radon.readthedocs.io/en/latest/) is a Python static analysis tool that computes quantitative metrics to assess code quality, readability, and structural complexity.
- The LLM *names* those smells in English and proposes a patch.
- **pytest + radon** decide whether the patch ships. The model does not get a vote.
### 👃 Smell · *The Symptom*
> **What it is:** Valid, executable code signaling friction ahead.  
> **Intuition:** A squeaky brake pad—the car stops, but wear is compounding.  
> **Payoff:** **Friction.** Adds cognitive overhead on every read/touch.

---

### ⚠️ Anti-pattern · *The Flawed Strategy*
> **What it is:** A recurring, broken blueprint disguised as a fix.  
> **Intuition:** Slapping tape over a check-engine light to clear inspection.  
> **Payoff:** **Risk.** Silently breeds regressions and reproducibility gaps.

---

### 💳 Debt · *The Compounding Ledger*
> **What it is:** The financial balance of delayed structural upkeep.  
> **Intuition:** A credit card balance—ignoring principal balloons the bill.  
> **Payoff:**
> • Quick win
> • Big rewrite

---

**Case study (no git clone).** We do **not** pull random GitHub. We write two files
that look like a real SaaS:

1. **PayFlow** — `checkout.py` charges invoices and issues refunds. Tests pin the public API.
2. **Churn trainer** — `train_churn.py` is a notebook-style sklearn job with leakage and no seed.

**Today you will**

1. Talk to an LLM safely (OpenRouter). Default OpenAI slug: `openai/gpt-5.6-luna`.
2. Parse code with an AST so smells are *queries on a tree*, not vibes.
3. Measure with PyExamine and MLScent; watch every tool command and every LLM reply.
4. Run a cooperative team: **auditor → planner → refactorer → QA** (no LLM in QA).
5. See the same team as **LangGraph** , then as **Deep Agents**.
6. Triage a real backlog and ship `TECH_DEBT_REPORT.md`.

Never paste a key into a cell.

![Key takeaway: tools measure, the LLM interprets, a gate decides](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgVFtEZXRlcm1pbmlzdGljIHRvb2xzPGJyLz5QeUV4YW1pbmUgwrcgTUxTY2VudCDCtyByYWRvbiDCtyBweXRlc3RdIC0tPiBMW0xMTSBpbnRlcnByZXRzPGJyLz5uYW1lcyBzbWVsbHMgwrcgcHJvcG9zZXMgYSBwYXRjaF0KICBMIC0tPiBHe0dhdGU8YnIvPnN5bnRheCDCtyB0ZXN0cyDCtyDOlCBDQ30KICBHIC0tPnxwYXNzfCBTW1NoaXBdCiAgRyAtLT58ZmFpbCArIHdoeXwgTA==)

**Figure.** The whole workshop in one loop.

# Part 0 · Setup + beginner basics  ·  0:00–0:25
## 0.1 · Install
Ignore Colab *Restart session* and the `google-auth` warning.

Two detector packages you are installing, and what they really are:

- **[PyExamine](https://github.com/KarthikShivasankar/python_smells_detector)** (`code-quality-analyzer`, MSR 2025) measures *code / structural / architectural* smells.
- **[MLScent](https://github.com/KarthikShivasankar/ml_smells_detector)** (`ml-code-smell-detector`, CAIN 2025) measures *ML anti-patterns* (leakage, seeds, eval mode).

Both are *static* analyzers: they parse source. They never train a model and they never run your tests.


In [1]:
%pip install -q openai radon pylint pytest pandas langgraph deepagents langchain-openai code-quality-analyzer ml-code-smell-detector
print("toolbox installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.7/540.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 909.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43

## 0.2 · Key + runtime
`OPENROUTER_API_KEY` + optional `LLM_MODEL` (any [OpenRouter slug](https://openrouter.ai/models)).

Load order: `os.environ` / `.env` → Colab Secrets → hidden prompt. Never paste a key into a cell.
Local: copy `.env.example` to `.env`.

This cell *is* the library (`llm`, `extract_json`, `switch_model`, `write_case_study`, `run_traced`, …).

**OpenAI model for this workshop:** only `openai/gpt-5.6-luna`.
That slug needs `max_completion_tokens` (not `max_tokens`) and prefers `reasoning_effort="low"`
so the visible reply is JSON, not an empty reasoning dump. `llm()` already does this.


In [4]:
# Self-contained runtime — OpenRouter via openai.
# This cell is the whole toolbox. No workshop_lib.py required on Colab.
"""Shared OpenRouter + openai-client helpers for Workshop 4.

The notebook inlines this module so Colab and local Jupyter share one brain.
Students never paste a key into a cell: credentials come from os.environ, a
.env file (this folder or parents), Colab Secrets, or a hidden prompt.

Default OpenAI model is openai/gpt-5.6-luna. To use another *provider*, set
LLM_MODEL to that OpenRouter slug and call switch_model() / reset_client().
"""
from __future__ import annotations

import json
import os
import pathlib
import re
import subprocess
import sys
from typing import Any
# OPENROUTER_API_KEY = "sk ----xxxx"
OPENROUTER_URL = "https://openrouter.ai/api/v1"
DEFAULT_MODEL = "openai/gpt-5.6-luna"
USAGE: dict[str, int] = {"calls": 0, "in": 0, "out": 0}
TRACE = True

_CLIENT = None


def dotenv_paths() -> list[pathlib.Path]:
    """Places a student might put OPENROUTER_API_KEY / LLM_MODEL.

    Works when this file is imported *and* when the same source is exec'd
    inside a notebook cell (no ``__file__``). Walks a few parents so a
    ``.env`` next to the repo or in Colab ``/content`` is still found after
    ``os.chdir(WORKDIR)``.
    """
    paths = [pathlib.Path("/content/.env")]
    cur = pathlib.Path.cwd().resolve()
    for _ in range(5):
        paths.append(cur / ".env")
        if cur.parent == cur:
            break
        cur = cur.parent
    try:
        here = pathlib.Path(__file__).resolve().parent
        paths.append(here / ".env")
        paths.append(here.parent / ".env")
    except NameError:
        pass
    seen: set[pathlib.Path] = set()
    out: list[pathlib.Path] = []
    for p in paths:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def load_dotenv() -> None:
    for candidate in dotenv_paths():
        if not candidate.is_file():
            continue
        for line in candidate.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            name, value = line.split("=", 1)
            os.environ.setdefault(name.strip(), value.strip().strip("'\""))


def load_credentials() -> str:
    """os.environ / .env first, then Colab Secrets, then getpass."""
    load_dotenv()
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"]

    try:
        from google.colab import userdata  # type: ignore
        for name in ("OPENROUTER_API_KEY", "LLM_MODEL"):
            try:
                val = userdata.get(name)
                if val:
                    os.environ[name] = str(val).strip()
            except Exception:
                pass
        if os.environ.get("OPENROUTER_API_KEY"):
            return os.environ["OPENROUTER_API_KEY"]
    except Exception:
        pass

    import getpass
    key = getpass.getpass("OpenRouter API key (hidden): ").strip()
    os.environ["OPENROUTER_API_KEY"] = key
    return key


def model_name() -> str:
    load_dotenv()
    return os.environ.get("LLM_MODEL") or DEFAULT_MODEL


def is_reasoning_model(name: str | None = None) -> bool:
    """GPT-5 family (including luna), o-series, and :thinking slugs.

    These models want max_completion_tokens, not max_tokens, and often
    reject temperature. reasoning_effort is set separately in llm().
    """
    n = (name or model_name()).lower()
    # Do not match the provider prefix "thinkingmachines/…" — that is not a reasoning slug.
    if "gpt-5" in n or ":thinking" in n:
        return True
    tokens = n.replace("/", " ").replace(":", " ").replace("-", " ").split()
    return any(tok in tokens for tok in ("o1", "o3", "o4", "reasoner"))


def record_usage(usage: Any, meter: dict[str, int] | None = None) -> None:
    """OpenRouter often returns usage=None — never dereference blindly."""
    meter = meter if meter is not None else USAGE
    if usage is None:
        return
    meter["in"] += int(getattr(usage, "prompt_tokens", 0) or 0)
    meter["out"] += int(getattr(usage, "completion_tokens", 0) or 0)


def extract_json(text: str) -> list:
    """Always a list. Empty / {} / prose without JSON → [].

    Prefers a ```json fence, then the first {...} or [...] blob. This is
    why a free model that replies in essays produces 0 findings: there is
    nothing here to parse.
    """
    if not text or not str(text).strip():
        return []
    blob = str(text)
    fenced = re.search(r"```(?:json)?\s*\n(.*?)```", blob, re.DOTALL)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(blob)
    for src in candidates:
        match = re.search(r"\[.*\]|\{.*\}", src, re.DOTALL)
        if not match:
            continue
        try:
            data = json.loads(match.group(0))
        except json.JSONDecodeError:
            tight = re.search(r'\{\s*"findings"\s*:\s*\[.*?\]\s*\}', src, re.DOTALL)
            if not tight:
                continue
            try:
                data = json.loads(tight.group(0))
            except json.JSONDecodeError:
                continue
        parsed = _as_findings(data)
        if parsed:
            return parsed
    return []


def _as_findings(data: Any) -> list:
    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]
    if isinstance(data, dict):
        findings = data.get("findings", data)
        if isinstance(findings, list):
            return [x for x in findings if isinstance(x, dict)]
        if isinstance(findings, dict) and (findings.get("smell") or findings.get("name")):
            return [findings]
    return []


def findings_from_evidence(evidence: str) -> list[dict]:
    """Last-resort parser: turn PyExamine / radon / MLScent text into findings.

    Used when the LLM replies with prose instead of JSON so the workshop
    blackboard is never empty after the tools actually found smells.
    """
    if not evidence:
        return []
    out: list[dict] = []
    needles = (
        "long method", "large class", "feature envy", "cyclomatic",
        "god object", "dead code", "duplicate", "data leakage",
        "missing random seed", "random seed", "eval mode", "shotgun",
        "temporary field", "divergent change", "speculative",
        "broad-exception", "leakage",
    )
    for line in evidence.splitlines():
        text = line.strip().lstrip("-").strip()
        if len(text) < 8:
            continue
        low = text.lower()
        hit = next((n for n in needles if n in low), None)
        if hit:
            out.append({
                "smell": hit.replace("_", " ").title(),
                "location": text[:140],
                "severity": "medium",
                "why": text[:280],
                "fix": "Split the long path into named helpers, or restore the missing ML guard the tool named.",
                "source": "tool-fallback",
            })
    for match in re.finditer(r"([A-Za-z_][\w.]*)\s+-\s+([C-F])\s+\((\d+)\)", evidence):
        out.append({
            "smell": "High Cyclomatic Complexity",
            "location": match.group(1),
            "severity": "high" if match.group(2) >= "D" else "medium",
            "why": f"radon rank {match.group(2)} complexity {match.group(3)}",
            "fix": "Extract each decision branch (region, inventory, payment) into a helper.",
            "source": "radon-fallback",
        })
    seen: set[tuple[str, str]] = set()
    unique: list[dict] = []
    for row in out:
        key = (row["smell"], row["location"][:80])
        if key not in seen:
            seen.add(key)
            unique.append(row)
    return unique[:8]


def snap_label(reply: str, labels: list[str], default: str = "code") -> str:
    if not reply or not str(reply).strip():
        return default
    token = str(reply).lower().strip().split()[0].strip(".,:;")
    return token if token in labels else default


def make_client(api_key: str | None = None):
    from openai import OpenAI
    key = api_key or os.environ.get("OPENROUTER_API_KEY")
    if not key:
        raise RuntimeError("OPENROUTER_API_KEY is not set. Run load_credentials() first.")
    return OpenAI(
        base_url=os.environ.get("OPENROUTER_BASE_URL", OPENROUTER_URL),
        api_key=key,
        default_headers={
            "HTTP-Referer": "https://github.com/KarthikShivasankar/LLMA4SE-Workshop4",
            "X-Title": "LLMA4SE Workshop 4",
        },
    )


def get_client():
    global _CLIENT
    if _CLIENT is None:
        _CLIENT = make_client()
    return _CLIENT


def reset_client() -> None:
    """Drop the cached client after you change LLM_MODEL or the API key."""
    global _CLIENT
    _CLIENT = None


def switch_model(slug: str) -> str:
    """Change the OpenRouter brain for every later llm() call.

    If you stay on OpenAI, use only openai/gpt-5.6-luna in this workshop.
    Other *providers* are fine: anthropic/..., google/..., etc.
    """
    os.environ["LLM_MODEL"] = slug.strip()
    reset_client()
    print(f"LLM switched to {model_name()}")
    print("  A new OpenAI() client will be created on the next llm() call.")
    return model_name()


def set_trace(on: bool) -> None:
    global TRACE
    TRACE = bool(on)
    print("trace", "ON — every tool and LLM call will print what it did" if TRACE else "OFF")


def _reply_text(resp: Any) -> str:
    choices = getattr(resp, "choices", None) or []
    if not choices:
        return ""
    msg = choices[0].message
    content = getattr(msg, "content", None)
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text") or "")
            elif hasattr(block, "text"):
                parts.append(block.text or "")
        return "".join(parts).strip()
    if content:
        return str(content).strip()
    # Some GPT-5 replies put visible text on refusal / reasoning fields.
    for attr in ("refusal", "reasoning"):
        extra = getattr(msg, attr, None)
        if extra:
            return str(extra).strip()
    return ""


def llm(
    user_prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    max_new_tokens: int = 1024,
    temperature: float = 0.2,
    json_mode: bool = False,
    client=None,
    model: str | None = None,
) -> str:
    """One OpenRouter chat completion via the official openai client.

    Behind the scenes (when TRACE is on):
      1. pick the slug from LLM_MODEL (default openai/gpt-5.6-luna)
      2. POST chat.completions.create to https://openrouter.ai/api/v1
      3. GPT-5 family gets max_completion_tokens + reasoning_effort=low
      4. if the first call fails (json_mode / temperature), retry stripped
    """
    cli = client or get_client()
    name = model or model_name()
    kwargs: dict[str, Any] = {
        "model": name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }
    if is_reasoning_model(name):
        kwargs["max_completion_tokens"] = max(max_new_tokens * 4, 4000)
        if "gpt-5" in name.lower():
            kwargs["reasoning_effort"] = "low"
    else:
        kwargs["max_tokens"] = max_new_tokens
        kwargs["temperature"] = temperature
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    if TRACE:
        print("\n" + "=" * 60)
        print("BEHIND THE SCENES · LLM")
        print(f"  model   : {name}")
        print(f"  json    : {json_mode}")
        print(f"  kwargs  : {[k for k in kwargs if k != 'messages']}")
        print(f"  system  : {system_prompt[:180].replace(chr(10), ' ')}")
        print(f"  user    : {user_prompt[:280].replace(chr(10), ' ')}…")

    try:
        resp = cli.chat.completions.create(**kwargs)
    except Exception as first:
        if TRACE:
            print(f"  first call failed: {type(first).__name__}: {first}")
            print("  retrying without temperature / response_format …")
        kwargs.pop("temperature", None)
        kwargs.pop("response_format", None)
        if "max_tokens" in kwargs:
            kwargs["max_completion_tokens"] = max(int(kwargs.pop("max_tokens")) * 4, 4000)
        resp = cli.chat.completions.create(**kwargs)

    USAGE["calls"] += 1
    record_usage(getattr(resp, "usage", None))
    text = _reply_text(resp)
    if TRACE:
        print(f"  usage   : {getattr(resp, 'usage', None)}")
        print(f"  reply   : {len(text)} chars")
        print(text[:1500] if text else "  (empty visible content — extract_json will return [])")
        print("=" * 60 + "\n")
    return text


def cost_report(meter: dict[str, int] | None = None) -> None:
    m = meter if meter is not None else USAGE
    line = f"cost: {m['calls']} calls | {m['in']:,} in / {m['out']:,} out tokens"
    try:
        print(line)
    except UnicodeEncodeError:
        print(line.encode("ascii", "replace").decode("ascii"))


_MODULE_FALLBACK = {
    "radon": "radon",
    "pylint": "pylint",
    "analyze_code_quality": "code_quality_analyzer.main",
    "ml_smell_detector": "ml_code_smell_detector.cli",
}


def tool_bin(name: str) -> str:
    """Prefer the console script next to this interpreter (Colab / venv)."""
    folder = pathlib.Path(sys.executable).parent
    for candidate in (folder / name, folder / f"{name}.exe"):
        if candidate.exists():
            return str(candidate)
    return name


def tool_argv(name: str, *args: str) -> list[str]:
    """Build a command that still works when Windows blocks a console script.

    Order: ``python -m <package>`` when we know the module, else the console
    script next to this interpreter. Colab usually has both.
    """
    extra = [str(a) for a in args]
    module = _MODULE_FALLBACK.get(name)
    if module:
        return [sys.executable, "-m", module, *extra]
    return [tool_bin(name), *extra]


def run_traced(label: str, argv: list[str], timeout: int = 600, cwd: str | None = None):
    """Run a detector CLI and print the command, exit code, stdout, stderr."""
    print("\n" + "=" * 60)
    print(f"BEHIND THE SCENES · TOOL {label}")
    print(f"  command : {' '.join(argv)}")
    print(f"  cwd     : {cwd or os.getcwd()}")
    print(f"  python  : {sys.executable}")
    result = subprocess.run(argv, capture_output=True, text=True, timeout=timeout, cwd=cwd)
    print(f"  exit    : {result.returncode}")
    if result.stdout:
        print(f"  stdout  : {len(result.stdout)} chars")
        print(result.stdout[:1200])
    else:
        print("  stdout  : (empty)")
    if result.stderr:
        print(f"  stderr  : {len(result.stderr)} chars")
        print(result.stderr[:600])
    print("=" * 60 + "\n")
    return result


def openrouter_chat_model():
    """LangChain wrapper for Deep Agents only — still OpenRouter, same key."""
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(
        model=model_name(),
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=os.environ.get("OPENROUTER_BASE_URL", OPENROUTER_URL),
    )


# ---------------------------------------------------------------------------
# Real workshop case study — PayFlow invoices + churn trainer
# (replaces cloning pallets/itsdangerous and pytorch/examples)
# ---------------------------------------------------------------------------

CHECKOUT_SRC = '''"""PayFlow — invoice charging and refunds for a small SaaS billing team.

This is the kind of module that grows in a startup: one service that
validates an invoice, checks inventory, charges a card, issues refunds,
and sends mail. The public API still works. Changing any one path is
expensive. That cost is technical debt.

Public API the tests pin down:
    Invoice, PaymentGateway, Mailer, OrderFulfillmentService
    OrderFulfillmentService.fulfill_order(...)
    OrderFulfillmentService.apply_refund(...)
"""
from __future__ import annotations

from dataclasses import dataclass, field

SUPPORTED_REGIONS = frozenset({"EU", "US", "UK"})
PAID = "paid"
FAILED = "failed"
OPEN = "open"
REFUNDED = "refunded"


@dataclass
class Invoice:
    invoice_id: str
    customer_email: str
    amount_due_cents: int
    amount_paid_cents: int = 0
    amount_refunded_cents: int = 0
    status: str = OPEN
    items: list = field(default_factory=list)


class PaymentGateway:
    def charge(self, cents: int, customer_email: str) -> str:
        if cents <= 0:
            raise ValueError("charge amount must be positive")
        handle = customer_email.split("@")[0]
        return f"ch_{cents}_{handle}"

    def refund(self, charge_id: str, cents: int) -> str:
        if cents <= 0:
            raise ValueError("refund amount must be positive")
        return f"re_{charge_id}_{cents}"


class Mailer:
    def __init__(self) -> None:
        self.sent: list[tuple[str, str]] = []

    def send(self, to: str, subject: str) -> None:
        self.sent.append((to, subject))


class OrderFulfillmentService:
    """God-ish service: validation, inventory, payment, refund, and mail."""

    def __init__(self, gateway: PaymentGateway, mailer: Mailer) -> None:
        self.gateway = gateway
        self.mailer = mailer
        self.charges: dict[str, str] = {}

    def _email_ok(self, email: str) -> bool:
        return bool(email) and "@" in email and "." in email.split("@")[-1]

    def fulfill_order(
        self,
        invoice: Invoice,
        card_ok: bool,
        inventory_ok: bool,
        region: str,
    ) -> Invoice:
        if not invoice.invoice_id:
            invoice.status = FAILED
            return invoice
        if not self._email_ok(invoice.customer_email):
            invoice.status = FAILED
            return invoice
        if invoice.amount_due_cents <= 0:
            invoice.status = FAILED
            return invoice
        if not invoice.items:
            invoice.status = FAILED
            return invoice
        if region not in SUPPORTED_REGIONS:
            invoice.status = FAILED
            self.mailer.send(invoice.customer_email, "unsupported region")
            return invoice
        if not inventory_ok:
            invoice.status = FAILED
            self.mailer.send(invoice.customer_email, "out of stock")
            return invoice
        if not card_ok:
            invoice.status = FAILED
            self.mailer.send(invoice.customer_email, "card declined")
            return invoice
        if region == "EU" and not invoice.customer_email.endswith(".eu") and "vat" not in str(invoice.items):
            # Extra EU branch so cyclomatic complexity is real, not a toy `if`.
            invoice.items = list(invoice.items) + ["vat-pending"]
        if region == "US" and invoice.amount_due_cents > invoice.amount_paid_cents:
            pass
        if region == "UK" and invoice.status == OPEN:
            pass
        try:
            charge_id = self.gateway.charge(invoice.amount_due_cents, invoice.customer_email)
            self.charges[invoice.invoice_id] = charge_id
            invoice.amount_paid_cents = invoice.amount_due_cents
            invoice.status = PAID
            self.mailer.send(invoice.customer_email, "invoice paid")
        except Exception:
            invoice.status = FAILED
            self.mailer.send(invoice.customer_email, "payment error")
        return invoice

    def apply_refund(self, invoice: Invoice, refund_cents: int) -> Invoice:
        if not self._email_ok(invoice.customer_email):
            raise ValueError("invalid customer")
        if invoice.status not in {PAID, REFUNDED}:
            raise ValueError("invoice is not refundable")
        if refund_cents <= 0:
            raise ValueError("refund must be positive")
        # Load-bearing check — QA sabotage flips `>` to `<`.
        if invoice.amount_refunded_cents + refund_cents > invoice.amount_paid_cents:
            raise ValueError("refund exceeds amount paid")
        charge_id = self.charges.get(invoice.invoice_id)
        if not charge_id:
            raise ValueError("no charge on file")
        self.gateway.refund(charge_id, refund_cents)
        invoice.amount_refunded_cents += refund_cents
        if invoice.amount_refunded_cents == invoice.amount_paid_cents:
            invoice.status = REFUNDED
        self.mailer.send(invoice.customer_email, "refund issued")
        return invoice
'''

CHECKOUT_TESTS = '''"""Behaviour contract for PayFlow. Gate 2 runs this file. No LLM here."""
import pytest
from checkout import Invoice, Mailer, OrderFulfillmentService, PaymentGateway


def _service():
    return OrderFulfillmentService(PaymentGateway(), Mailer())


def _open_invoice(**overrides):
    data = dict(
        invoice_id="inv_100",
        customer_email="buyer@shop.example",
        amount_due_cents=2500,
        items=["sku-hat"],
    )
    data.update(overrides)
    return Invoice(**data)


def test_fulfill_happy_path_marks_paid_and_mails():
    svc = _service()
    inv = svc.fulfill_order(_open_invoice(), card_ok=True, inventory_ok=True, region="US")
    assert inv.status == "paid"
    assert inv.amount_paid_cents == 2500
    assert svc.mailer.sent[-1][1] == "invoice paid"


def test_fulfill_rejects_bad_email_without_charging():
    svc = _service()
    inv = svc.fulfill_order(_open_invoice(customer_email="not-an-email"), True, True, "US")
    assert inv.status == "failed"
    assert svc.charges == {}


def test_fulfill_out_of_stock_mails_customer():
    svc = _service()
    inv = svc.fulfill_order(_open_invoice(), card_ok=True, inventory_ok=False, region="UK")
    assert inv.status == "failed"
    assert svc.mailer.sent[-1][1] == "out of stock"


def test_apply_refund_rejects_over_refund():
    svc = _service()
    inv = svc.fulfill_order(_open_invoice(), True, True, "EU")
    with pytest.raises(ValueError, match="exceeds"):
        svc.apply_refund(inv, inv.amount_paid_cents + 1)


def test_full_refund_marks_refunded():
    svc = _service()
    inv = svc.fulfill_order(_open_invoice(), True, True, "US")
    inv = svc.apply_refund(inv, inv.amount_paid_cents)
    assert inv.status == "refunded"
    assert inv.amount_refunded_cents == inv.amount_paid_cents
'''

CHURN_SRC = '''"""Customer-churn trainer that looks like a real notebook export.

MLScent walks this file as an AST. It does not import sklearn or pandas.
The smells we planted are the ones that show up in production training jobs:

* no random seed — two reruns, two different splits, two different models
* StandardScaler.fit on the *full* frame before the split — test rows leak
  into the mean / variance the model is trained with
* accuracy printed on the *training* split and treated as the reported score
"""
from __future__ import annotations

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def load_customers() -> pd.DataFrame:
    rows = []
    for i in range(80):
        rows.append({
            "tenure_months": i % 24,
            "tickets_opened": i % 7,
            "monthly_spend": 20 + (i % 15),
            "churned": int(i % 5 == 0),
        })
    return pd.DataFrame(rows)


def train_churn_model():
    frame = load_customers()
    features = frame.drop(columns=["churned"])
    labels = frame["churned"]

    # Leakage: fit the scaler on every row, *then* split.
    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)

    # No random_state — the split is not reproducible.
    x_train, x_test, y_train, y_test = train_test_split(scaled, labels, test_size=0.25)

    model = LogisticRegression()
    model.fit(x_train, y_train)

    # Train score reported as if it were hold-out performance.
    train_score = model.score(x_train, y_train)
    print("reported_score", train_score)
    _ = x_test, y_test
    return model


if __name__ == "__main__":
    train_churn_model()
'''


def write_case_study(root: str | pathlib.Path | None = None) -> dict[str, str]:
    """Write the PayFlow + churn case onto disk. No git clone required."""
    root = pathlib.Path(root or "cases").resolve()
    payflow = root / "payflow"
    churn = root / "churn"
    payflow.mkdir(parents=True, exist_ok=True)
    churn.mkdir(parents=True, exist_ok=True)
    checkout = payflow / "checkout.py"
    tests = payflow / "test_checkout.py"
    trainer = churn / "train_churn.py"
    checkout.write_text(CHECKOUT_SRC, encoding="utf-8")
    tests.write_text(CHECKOUT_TESTS, encoding="utf-8")
    trainer.write_text(CHURN_SRC, encoding="utf-8")
    return {
        "CASE_PY": str(checkout),
        "CASE_TESTS": str(tests),
        "ML_CASE_DIR": str(churn),
        "ML_CASE_PY": str(trainer),
        "CASE_ROOT": str(root),
        "PAYFLOW_DIR": str(payflow),
    }


key = load_credentials()
print("key loaded, ends with …" + key[-4:])
print("model:", model_name())
print("OpenAI default for this workshop is openai/gpt-5.6-luna")
print("TRACE is", TRACE, "— every tool and LLM call will print what it did")

key loaded, ends with …7a4a
model: cohere/north-mini-code:free
OpenAI default for this workshop is openai/gpt-5.6-luna
TRACE is True — every tool and LLM call will print what it did


## 0.3 · How to change the LLM (one function)

The brain is **one environment variable**. After you change it you must drop the cached
client, otherwise the next call still uses the old slug.

```python
# Stay on OpenAI → use ONLY this slug in this workshop
switch_model("openai/gpt-5.6-luna")

# Other providers (not OpenAI) via the same OpenRouter key
# switch_model("anthropic/claude-sonnet-4")
# switch_model("google/gemini-2.5-flash")
```

`switch_model` is `os.environ["LLM_MODEL"] = slug` plus `reset_client()`.
Run the cell below as-is (luna) so everyone is on the same brain. To try another
*provider*, uncomment one line, re-run this cell, then re-run the hello call.


In [5]:
# How to change the brain: switch_model("provider/slug") then the next llm() uses it.
# OpenAI for this workshop is ONLY openai/gpt-5.6-luna. Other lines are other providers.
# We probe in order so a missing OpenAI quota still lets the notebook run.

CANDIDATES = [
    # Free slugs first so a dry-run / CI pass does not spend OpenAI quota.
    "cohere/north-mini-code:free",
    "thinkingmachines/inkling:free",
    "thinkingmachines/inkling-small:free",
    "poolside/laguna-s-2.1:free",
    "nvidia/nemotron-3.5-lightning:free",
    # Only OpenAI slug for this workshop — uncomment / move to the top in class:
    # "openai/gpt-5.6-luna",
]
ready = None
for slug in CANDIDATES:
    print("probing", slug)
    switch_model(slug)
    try:
        probe = llm("Reply with the single word: ok", max_new_tokens=24)
    except Exception as e:
        print("  failed:", type(e).__name__, e)
        reset_client()
        continue
    if probe.strip():
        ready = slug
        print("ready — later cells use", slug)
        print("probe:", probe[:200])
        break
    print("  empty visible reply — next slug")
    reset_client()
if not ready:
    raise RuntimeError("No OpenRouter slug returned text. Re-run §0.2 and check OPENROUTER_API_KEY.")
print("active model:", model_name())
print("reasoning-family (max_completion_tokens)?", is_reasoning_model())

probing cohere/north-mini-code:free
LLM switched to cohere/north-mini-code:free
  A new OpenAI() client will be created on the next llm() call.

BEHIND THE SCENES · LLM
  model   : cohere/north-mini-code:free
  json    : False
  kwargs  : ['model', 'max_tokens', 'temperature']
  system  : You are a helpful assistant.
  user    : Reply with the single word: ok…
  usage   : CompletionUsage(completion_tokens=24, prompt_tokens=13, total_tokens=37, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=23, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0})
  reply   : 92 chars
The user asks: "Reply with the single word: ok". So we need to respond with exactly "ok". No

ready 

In [6]:
switch_model("openai/gpt-5.6-luna")

LLM switched to openai/gpt-5.6-luna
  A new OpenAI() client will be created on the next llm() call.


'openai/gpt-5.6-luna'

## 0.4 · Your first LLM call

With **BEHIND THE SCENES · LLM TRACE** on you will see the model, the kwargs, a preview of the prompt, and the raw reply.
A sentence back means the key, the URL, and the slug all work.
Silence (`reply: 0 chars`) is a *model* problem — switch back to `openai/gpt-5.6-luna`.


In [7]:
print(llm("In one sentence: what is a code smell in a billing service?"))
cost_report()


BEHIND THE SCENES · LLM
  model   : openai/gpt-5.6-luna
  json    : False
  kwargs  : ['model', 'max_completion_tokens', 'reasoning_effort']
  system  : You are a helpful assistant.
  user    : In one sentence: what is a code smell in a billing service?…
  usage   : CompletionUsage(completion_tokens=45, prompt_tokens=30, total_tokens=75, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=6e-05, is_byok=False, cost_details={'upstream_inference_cost': 6e-05, 'upstream_inference_prompt_cost': 6e-06, 'upstream_inference_completions_cost': 5.4e-05})
  reply   : 232 chars
A code smell in a billing service is a design or implementation warning—such as duplicated pricing logic, tangled payment workflows, or hard-coded rates—that may indicate deeper reliability

## 0.5 · Beginner map — six words we will keep using

| Idea | Walk away with |
|---|---|
| **LLM call** | Text in, tokens out. `openai.OpenAI(base_url=OpenRouter)` is a client. `LLM_MODEL` picks the brain. |
| **Tokens** | Characters ≠ tokens. OpenRouter may omit `usage`. Temperature is a *sampling* knob, not intelligence. GPT-5.6 Luna often ignores temperature. |
| **Tool** | A deterministic function the model did *not* write (radon, PyExamine, pytest). Tools measure. |
| **Agent** | Four slots: **role** (system prompt) · **brain** (`llm`) · **tools** · **contract** (JSON / one code fence / exit code). |
| **Gate** | A check with **no LLM**. If the gate fails, the patch did not happen. |
| **Workflow** | Several agents sharing a **blackboard**. If it is not on the board, it did not happen. |



## 0.6 · Tokens
The string `fulfill_order`   usually 2–4 tokens. `usage is None` is normal, not a bug.
A refactorer wants **reproducible** patches: keep temperature low (`0.0`–`0.2`) on models that accept it.
`openai/gpt-5.6-luna` uses `reasoning_effort="low"` instead.


In [8]:
probe = "def fulfill_order(self, invoice, card_ok, inventory_ok, region):"
kwargs = {"model": model_name(), "messages": [{"role": "user", "content": f"Repeat exactly: {probe}"}]}
if is_reasoning_model():
    kwargs["max_completion_tokens"] = 80
    kwargs["reasoning_effort"] = "low"
else:
    kwargs["max_tokens"] = 80
r = make_client().chat.completions.create(**kwargs)
u = r.usage
print("chars sent", len(probe))
if u is None:
    print("usage: None (OpenRouter omitted it — fine)")
else:
    print("tokens in/out", u.prompt_tokens, u.completion_tokens)
print("reply:", (r.choices[0].message.content or "")[:120])

chars sent 64
tokens in/out 24 19
reply: def fulfill_order(self, invoice, card_ok, inventory_ok, region):


## 0.7 · Workspace + clock
We `chdir` into a workshop folder so reports do not clutter your home directory.
`WORKSHOP_ROOT` remembers where `debtbuster/` lives *before* that chdir.
We also put the current interpreter’s `bin` / `Scripts` on `PATH` so `analyze_code_quality`
and `ml_smell_detector` resolve after the pip cell.


In [9]:
import os, pathlib, sys, time, json, re, ast, subprocess, shutil, tempfile

os.environ["PATH"] = str(pathlib.Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
WORKSHOP_ROOT = pathlib.Path.cwd().resolve()
for p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path("/content/LLMA4SE-Workshop4")):
    if (p / "debtbuster" / "pyproject.toml").is_file():
        WORKSHOP_ROOT = p.resolve()
        break
WORKDIR = (pathlib.Path("/content/workshop") if pathlib.Path("/content").exists()
           else pathlib.Path("./workshop")).resolve()
WORKDIR.mkdir(exist_ok=True)
os.chdir(WORKDIR)
print("working in", os.getcwd())
print("workshop root", WORKSHOP_ROOT)

T0 = time.time()
SCHEDULE = {"Part 0": 25, "Part 1": 70, "Part 2": 110, "Part 3": 150, "Part 4": 175}

def pit_stop(part: str) -> None:
    mins = (time.time() - T0) / 60
    drift = SCHEDULE[part] - mins
    mood = ("ahead" if drift > 5 else "on time" if drift > -5 else "behind — skip Deep Agents if needed")
    print(f"{mins:5.0f} min elapsed | {part} budget {SCHEDULE[part]} | {mood}")

pit_stop("Part 0")

working in /content/workshop
workshop root /content
    0 min elapsed | Part 0 budget 25 | ahead


## 0.8 · The case study — PayFlow invoices + a churn trainer

| Path | Story | What we will do to it |
|---|---|---|
| `cases/payflow/checkout.py` | SaaS billing: validate → inventory → charge card → refund → email | PyExamine + radon + a refactor team |
| `cases/payflow/test_checkout.py` | Public-API contract (`fulfill_order`, `apply_refund`) | Gate 2. Sabotage flips the refund cap. |
| `cases/churn/train_churn.py` | sklearn churn job: scaler fit *before* the split, no seed, train score reported | MLScent |

`write_case_study()` just writes files. No network.


In [10]:
paths = write_case_study(WORKDIR / "cases")
CASE_PY = paths["CASE_PY"]
CASE_TESTS = paths["CASE_TESTS"]
ML_CASE_DIR = paths["ML_CASE_DIR"]
ML_CASE_PY = paths["ML_CASE_PY"]
PAYFLOW_DIR = paths["PAYFLOW_DIR"]
print("PayFlow module :", CASE_PY)
print("PayFlow tests  :", CASE_TESTS)
print("Churn trainer  :", ML_CASE_PY)
print()
print("--- checkout.py (first 25 lines) ---")
print("\n".join(open(CASE_PY, encoding="utf-8").read().splitlines()[:25]))
print()
r = subprocess.run([sys.executable, "-m", "pytest", CASE_TESTS, "-q"], capture_output=True, text=True)
print(r.stdout or r.stderr)
print("pytest exit", r.returncode, "(0 = the case study is a working product)")

PayFlow module : /content/workshop/cases/payflow/checkout.py
PayFlow tests  : /content/workshop/cases/payflow/test_checkout.py
Churn trainer  : /content/workshop/cases/churn/train_churn.py

--- checkout.py (first 25 lines) ---
"""PayFlow — invoice charging and refunds for a small SaaS billing team.

This is the kind of module that grows in a startup: one service that
validates an invoice, checks inventory, charges a card, issues refunds,
and sends mail. The public API still works. Changing any one path is
expensive. That cost is technical debt.

Public API the tests pin down:
    Invoice, PaymentGateway, Mailer, OrderFulfillmentService
    OrderFulfillmentService.fulfill_order(...)
    OrderFulfillmentService.apply_refund(...)
"""
from __future__ import annotations

from dataclasses import dataclass, field

SUPPORTED_REGIONS = frozenset({"EU", "US", "UK"})
PAID = "paid"
FAILED = "failed"
OPEN = "open"
REFUNDED = "refunded"


@dataclass
class Invoice:

.....                             

# Part 1 · Parse, then measure  ·  0:25–1:10
A **bug** makes tests fail. A **smell** is legal code that costs interest. An **anti-pattern**
is a recurring wrong solution (especially in training loops). Detectors do not “read code like
a human.” They **parse** it. If you skip this, the CSV looks like magic.

## 1.1 · Why parsers exist
Source code is a string. A parser turns it into a tree of nodes (`FunctionDef`, `If`, `Call`).
Smells are *queries on that tree*: “any function with more than N `If` nodes,”
“any `StandardScaler.fit` that happens before `train_test_split`.”
Without a tree you are grepping, and grepping lies — a comment that says `fit_transform`
is not a leak.

Gate 1 later is `ast.parse` (syntax). Cyclomatic complexity *starts* as “count decision nodes.”

![Abstract syntax tree](https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c7/Abstract_syntax_tree_for_Euclidean_algorithm.svg/960px-Abstract_syntax_tree_for_Euclidean_algorithm.svg.png)

**Figure.** An AST for a tiny algorithm. PyExamine / MLScent / `ast.parse` do this to *your*
PayFlow file. Source: Wikimedia Commons, *Abstract syntax tree for Euclidean algorithm* (CC BY-SA).

![Compiler pipeline](https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6b/Compiler.svg/960px-Compiler.svg.png)

**Figure.** A compiler also lexes, parses, then analyses. Our detectors **stop at analysis** —
they never emit a binary and they never run `fulfill_order`. Source: Wikimedia Commons, *Compiler*.


In [11]:
src = '''
def fulfill_order(invoice, card_ok, inventory_ok, region):
    if not invoice.invoice_id:
        return "failed"
    if not inventory_ok:
        return "failed"
    if not card_ok:
        return "failed"
    if region not in ("EU", "US", "UK"):
        return "failed"
    if region == "EU":
        invoice.items.append("vat-pending")
    return "paid"
'''
tree = ast.parse(src)
print("=== ast.dump of the fulfill_order tree (truncated) ===")
print(ast.dump(tree, indent=2)[:900])
print()
print("If nodes       :", sum(isinstance(n, ast.If) for n in ast.walk(tree)))
print("FunctionDef    :", sum(isinstance(n, ast.FunctionDef) for n in ast.walk(tree)))
print()
print("Each `if` is one extra path a test should cover. radon will count those paths.")
print("PyExamine asks a richer question: is fulfill_order a Long Method / Feature Envy / God Object?")

=== ast.dump of the fulfill_order tree (truncated) ===
Module(
  body=[
    FunctionDef(
      name='fulfill_order',
      args=arguments(
        args=[
          arg(arg='invoice'),
          arg(arg='card_ok'),
          arg(arg='inventory_ok'),
          arg(arg='region')]),
      body=[
        If(
          test=UnaryOp(
            op=Not(),
            operand=Attribute(
              value=Name(id='invoice', ctx=Load()),
              attr='invoice_id',
              ctx=Load())),
          body=[
            Return(
              value=Constant(value='failed'))]),
        If(
          test=UnaryOp(
            op=Not(),
            operand=Name(id='inventory_ok', ctx=Load())),
          body=[
            Return(
              value=Constant(value='failed'))]),
        If(
          test=UnaryOp(
            op=Not(),
            operand=Name(id='card_ok', ctx=Load())),
          body=[
            Return(
              value=Cons

If nodes       : 5
FunctionDef    : 1

Each

## 1.2 · Tree-sitter
CPython `ast` / **astroid** know Python *semantics* (scopes, what a name refers to).
That is why PyExamine and MLScent chose them.

**Tree-sitter** is the incremental, error-tolerant, multi-language sibling. Editors and GitHub
use it so a file you have not finished typing still highlights. Semgrep uses it to search
many languages. A broken file still gets a *partial* tree. `ast.parse` raises `SyntaxError`.

| | CPython `ast` / astroid | Tree-sitter |
|---|---|---|
| Job | Exact Python semantics | Incremental, error-tolerant, many languages |
| Who uses it here | PyExamine + MLScent | Editors, GitHub, Semgrep, other linters |
| Broken file | `SyntaxError` — skip / fail | Still builds a partial tree |
| Workshop | We **run** this | Prerequisite knowledge |

Instructor one-liner: *Tree-sitter is why VS Code still highlights a file you have not finished typing.
Our detectors chose astroid because they need Python-aware scopes, not a generic CST.*


## 1.3 · PyExamine — what it is, what it is not
**[PyExamine](https://github.com/KarthikShivasankar/python_smells_detector)** (MSR 2025) is a
multi-level smell detector for Python. Pip name: `code-quality-analyzer`. CLI: `analyze_code_quality`.

It does **not** run your program. It parses every `.py` file under a *directory* with **astroid**,
then asks three families of questions:

| Layer | Typical questions | Example on PayFlow |
|---|---|---|
| **Code** (18 + 5 cross-file) | Long Method? Feature Envy? Dead code? Duplicate blocks? | `fulfill_order` does validation + charge + mail |
| **Structural** (OO metrics) | High cyclomatic complexity, too many methods, low cohesion | many `if` branches in one method |
| **Architectural** | God object, cyclic imports, hub modules | one service owns the whole billing story |

Behind the scenes of the next cell:

1. We resolve `analyze_code_quality` next to *this* Python (Colab/venv), not a random PATH hit.
2. We run `--type code --output pyx` on the PayFlow **directory** (PyExamine wants a folder).
3. The tool writes `pyx.txt` (human) and `pyx.csv` (one row per smell).
4. `run_traced` prints the command, exit code, stdout, and stderr so a silent failure is visible.

What PyExamine **cannot** do: run `test_checkout.py`, decide a merge, or invent a missing smell
that is not in the AST.


## 1.4 · MLScent — what it is, what it is not
**[MLScent](https://github.com/KarthikShivasankar/ml_smells_detector)** (CAIN 2025) is a static
analyzer for *machine-learning* projects. Pip name: `ml-code-smell-detector`. CLI: `ml_smell_detector`.

It implements on the order of **76 detectors** across TensorFlow, PyTorch, scikit-learn,
Hugging Face, pandas, and numpy — plus general ML smells. It **never imports those libraries**.
A Colab CPU runtime can audit a training script without a GPU and without `pip install torch`.

Typical hits on a real trainer (and on our churn file):

| Smell | Why it hurts in production |
|---|---|
| Missing random seed | Two reruns, two splits, two models. Nobody can reproduce the paper. |
| Fit / transform before the split | Test rows leak into scaler mean/variance → optimistic scores. |
| Train score reported as the metric | You shipped a model that memorised the training customers. |
| Missing eval / `model.eval()` | Dropout stays on at test time (PyTorch). |

CLI: `ml_smell_detector analyze <file-or-dir> --output-dir <folder>`.
Reports: `analysis_report.txt` + `analysis_report.csv`.

What MLScent **cannot** do: prove the model trained correctly, or replace a hold-out test.


## 1.5 · Three eyes, one pipeline

| Tool | Measures | Cannot |
|---|---|---|
| **PyExamine** | Fowler-style + OO + architecture | Run tests; decide a merge |
| **MLScent** | ML anti-patterns. No sklearn/torch install needed | Prove the model is good |
| **radon** | Cheap cyclomatic complexity — later **gate 3** | Name a smell in English |

Pipeline: **1** parse (`ast`) → **2** measure (PyExamine / MLScent / radon) → **3** interpret (`llm`) → **4** decide (gates).


In [12]:
def run_radon(path: str) -> str:
    # radon cc -s : human table with rank A (simple) … F (untestable)
    result = run_traced("radon", tool_argv("radon", "cc", "-s", path))
    return result.stdout or result.stderr or "n/a"

def run_pylint(path: str, max_findings: int = 12) -> str:
    result = run_traced("pylint", tool_argv("pylint", path, "-f", "json", "--score", "n"))
    try:
        issues = json.loads(result.stdout or "[]")[:max_findings]
    except json.JSONDecodeError:
        return (result.stdout or result.stderr or "")[:1500]
    return "\n".join(f"L{i['line']}: [{i['symbol']}] {i['message']}" for i in issues) or "no findings"

def run_pyexamine(directory: str, smell_type: str = "code") -> str:
    '''PyExamine wants a *directory*. --output pyx writes pyx.txt + pyx.csv in cwd.'''
    out_base = str(WORKDIR / "pyx")
    argv = tool_argv(
        "analyze_code_quality", directory,
        "--type", smell_type, "--output", out_base,
        "--ignore", "tests", "venv", ".git", "__pycache__",
    )
    result = run_traced("pyexamine", argv)
    report = pathlib.Path(out_base + ".txt")
    if report.is_file():
        text = report.read_text(encoding="utf-8")
        print("PyExamine wrote", report, "·", len(text), "chars")
        return text[:4000]
    # Fallback: some versions print to stdout when the file path is odd
    combined = (result.stdout or "") + "\n" + (result.stderr or "")
    return combined[:4000] or "no PyExamine report — look at the TRACE block above"

def run_pyexamine_for_file(path: str) -> str:
    '''Auditor tools all take a file path; PyExamine still wants the folder.'''
    return run_pyexamine(str(pathlib.Path(path).parent))

print(run_pyexamine(PAYFLOW_DIR))


BEHIND THE SCENES · TOOL pyexamine
  command : /usr/bin/python3 -m code_quality_analyzer.main /content/workshop/cases/payflow --type code --output /content/workshop/pyx --ignore tests venv .git __pycache__
  cwd     : /content/workshop
  python  : /usr/bin/python3
  exit    : 0
  stdout  : 424 chars
Analyzing Code Smells...

Starting code smell analysis for directory: /content/workshop/cases/payflow
Successfully analyzed: /content/workshop/cases/payflow/test_checkout.py
Successfully analyzed: /content/workshop/cases/payflow/checkout.py

Code Smell Analysis Summary:
--------------------------
Files analyzed: 2
Files with errors: 0
Success rate: 100.0%
    
Text report generated and saved to /content/workshop/pyx.txt

  stderr  : 2449 chars
<frozen runpy>:130: RuntimeWarning: 'code_quality_analyzer.main' found in sys.modules after import of package 'code_quality_analyzer', but prior to execution of 'code_quality_analyzer.main'; this may result in unpredictable behaviour
2026-09-11 10:07

## 1.6 · radon — the cheap thermometer we will gate on
PyExamine is the rich report. radon is the **number** gate 3 will compare before vs after.
Rank **A** is simple. **C** and above means “this block is getting hard to test.”
Look at `fulfill_order` — that is the method with the nested payment / region branches.


In [13]:
print(run_radon(CASE_PY))
print()
print("--- pylint (style / bug-adjacent; not a merge gate) ---")
print(run_pylint(CASE_PY))
raw = subprocess.run(tool_argv("radon", "cc", "-j", CASE_PY), capture_output=True, text=True).stdout
data = json.loads(raw or "{}")
hard = [(b["name"], b["complexity"], b.get("rank"))
        for blocks in data.values() for b in blocks if b.get("rank", "A") >= "C"]
print("radon C+ blocks (the ones a gate might care about):", hard)


BEHIND THE SCENES · TOOL radon
  command : /usr/bin/python3 -m radon cc -s /content/workshop/cases/payflow/checkout.py
  cwd     : /content/workshop
  python  : /usr/bin/python3
  exit    : 0
  stdout  : 542 chars
/content/workshop/cases/payflow/checkout.py
    M 67:4 OrderFulfillmentService.fulfill_order - C (16)
    C 56:0 OrderFulfillmentService - B (8)
    M 116:4 OrderFulfillmentService.apply_refund - B (7)
    C 35:0 PaymentGateway - A (3)
    M 64:4 OrderFulfillmentService._email_ok - A (3)
    M 36:4 PaymentGateway.charge - A (2)
    M 42:4 PaymentGateway.refund - A (2)
    C 48:0 Mailer - A (2)
    C 25:0 Invoice - A (1)
    M 49:4 Mailer.__init__ - A (1)
    M 52:4 Mailer.send - A (1)
    M 59:4 OrderFulfillmentService.__init__ - A (1)


/content/workshop/cases/payflow/checkout.py
    M 67:4 OrderFulfillmentService.fulfill_order - C (16)
    C 56:0 OrderFulfillmentService - B (8)
    M 116:4 OrderFulfillmentService.apply_refund - B (7)
    C 35:0 PaymentGateway - A (3)
    M

## 1.7 · Model vs radon — why we do not gate on the LLM
Same file, two complexity numbers. radon is the ground truth. The model is guessing from text.
`extract_json` always returns a `list` — `{}`, empty, or prose become `[]`, not a crash.

This is the same failure mode that produced **saved 0 findings** in an earlier run:
the model talked, tokens were spent, and there was no JSON object to parse.


In [14]:
src = open(CASE_PY, encoding="utf-8").read()
guessed = extract_json(llm(
    "Return JSON {\"findings\":[{\"name\": fn, \"complexity\": int}]} for each function in this module.\n"
    f"```python\n{src}\n```",
    system_prompt="Compute McCabe cyclomatic complexity. Reply JSON only.",
    max_new_tokens=400, temperature=0.0,
))
radon_json = json.loads(subprocess.run(
    tool_argv("radon", "cc", "-j", CASE_PY), capture_output=True, text=True,
).stdout or "{}")
truth = {b["name"]: b["complexity"] for blocks in radon_json.values() for b in blocks}
print("radon truth :", truth)
print("model JSON  :", guessed)
wrong = 0
for g in guessed:
    name = (g.get("name") or "").split(".")[-1]
    if name in truth and g.get("complexity") != truth[name]:
        wrong += 1
        print(f"  miss {name}: model={g.get('complexity')} radon={truth[name]}")
print("wrong:", wrong, "of", len(truth), "— a metric you cannot trust to ±1 is a metric you cannot gate on.")


BEHIND THE SCENES · LLM
  model   : openai/gpt-5.6-luna
  json    : False
  kwargs  : ['model', 'max_completion_tokens', 'reasoning_effort']
  system  : Compute McCabe cyclomatic complexity. Reply JSON only.
  user    : Return JSON {"findings":[{"name": fn, "complexity": int}]} for each function in this module. ```python """PayFlow — invoice charging and refunds for a small SaaS billing team.  This is the kind of module that grows in a startup: one service that validates an invoice, checks inven…
  usage   : CompletionUsage(completion_tokens=634, prompt_tokens=1160, total_tokens=1794, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=516, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=1157, cached_tokens=0, video_tokens=0), cost=0.00105065, is_byok=False, cost_details={'upstream_inference_cost': 0.00105065, 'upstream_inference_prompt_cost': 

## 1.8 · MLScent on the churn trainer
We point the CLI at the **directory** (or the file). TRACE will show
`ml_smell_detector analyze … --output-dir …`. Look for seed / leakage / train-score language
in the report — those are the planted anti-patterns.


In [15]:
def run_mlscent(project_dir: str) -> str:
    out_dir = WORKDIR / "mlscent_out"
    out_dir.mkdir(exist_ok=True)
    argv = tool_argv("ml_smell_detector", "analyze", project_dir, "--output-dir", str(out_dir))
    result = run_traced("mlscent", argv)
    report = out_dir / "analysis_report.txt"
    if report.is_file():
        text = report.read_text(encoding="utf-8")
        print("MLScent wrote", report, "·", len(text), "chars")
        return text[:4000]
    combined = (result.stdout or "") + "\n" + (result.stderr or "")
    return combined[:4000] or "no MLScent report — look at the TRACE block above"

mls = run_mlscent(ML_CASE_DIR)
print(mls)
print()
low = mls.lower()
for needle in ("seed", "leak", "eval", "split", "train", "scaler"):
    print(f"  mention {needle!r}:", needle in low)


BEHIND THE SCENES · TOOL mlscent
  command : /usr/bin/python3 -m ml_code_smell_detector.cli analyze /content/workshop/cases/churn --output-dir /content/workshop/mlscent_out
  cwd     : /content/workshop
  python  : /usr/bin/python3
  exit    : 0
  stdout  : 142 chars
Analysis complete. Results written to /content/workshop/mlscent_out/analysis_report.txt and /content/workshop/mlscent_out/analysis_report.csv

  stderr  : 304 chars

Analyzing files:   0%|          | 0/1 [00:00<?, ?file/s]Skipping Hugging Face smell detection for /content/workshop/cases/churn/train_churn.py: 'transformers' not imported

Analyzing files: 100%|██████████| 1/1 [00:00<00:00,  4.16file/s]


MLScent wrote /content/workshop/mlscent_out/analysis_report.txt · 3664 chars
Analysis results for /content/workshop/cases/churn/train_churn.py:

Framework-Specific Smells:
- Column Selection Checker
  Framework: Pandas
  How to fix: Select necessary columns after importing DataFrame.
  Benefits: Clarifies data usage and imp

## 1.9 · Agent — four slots, tools first, JSON contract
An **agent** is not a framework. It is four slots:

1. **role** — the system prompt (“senior reviewer…”)
2. **brain** — `llm()` → OpenRouter → `openai/gpt-5.6-luna`
3. **tools** — PyExamine + radon (deterministic)
4. **contract** — JSON `{"findings":[...]}` so `extract_json` can parse it

`audit()` runs tools **first**, then shows the model the dump. That order matters:
if you ask the model first, it invents smells; if you ask it second, it *interprets*.



In [16]:
class Agent:
    def __init__(self, name, system_prompt, tools=None):
        self.name, self.system_prompt, self.tools = name, system_prompt, tools or {}

    def use_tools(self, *args):
        print(f"\n>>> {self.name} is calling {len(self.tools)} tool(s): {list(self.tools)}")
        bits = []
        for n, fn in self.tools.items():
            print(f"\n--- agent tool hook: {n} ---")
            try:
                dump = fn(*args)
                bits.append(f"=== {n} ===\n{dump}")
            except Exception as e:
                print("TOOL FAILED", n, type(e).__name__, e)
                bits.append(f"=== {n} FAILED: {type(e).__name__}: {e} ===")
        return "\n\n".join(bits)

    def think(self, prompt, **kw):
        print(f"\n>>> {self.name}.think() → llm()  json_mode={kw.get('json_mode')}")
        return llm(prompt, system_prompt=self.system_prompt, **kw)

AUDITOR_PROMPT = '''You are a senior reviewer for a SaaS billing codebase.
Use ONLY the tool evidence plus the source. Do not invent smells that are not supported.
Reply with JSON {"findings":[{"smell","location","severity","why","fix"}]} — at most 6.
Prefer Long Method, High Cyclomatic Complexity, God-like service, broad except, duplicated validation.'''

auditor = Agent("Code Auditor", AUDITOR_PROMPT,
                {"pyexamine": run_pyexamine_for_file, "radon": run_radon})

def audit(path: str) -> list[dict]:
    print("\n######## AUDIT START", path, "########")
    evidence = auditor.use_tools(path)
    print("\n--- evidence handed to the LLM (truncated) ---")
    print(evidence[:1500])
    raw = auditor.think(
        f"SOURCE ({path}):\n```python\n{open(path, encoding='utf-8').read()}\n```\nEVIDENCE:\n{evidence}\nJSON now.",
        max_new_tokens=900, temperature=0.1, json_mode=True,
    )
    findings = extract_json(raw)
    print("extract_json after json_mode:", len(findings), "findings")
    if not findings:
        print("JSON mode produced nothing parseable — retrying without json_mode")
        raw = auditor.think(
            f"SOURCE ({path}):\n```python\n{open(path, encoding='utf-8').read()[:4000]}\n```\n"
            f"EVIDENCE:\n{evidence}\nReply with ONLY a JSON object {{'findings':[...]}}.",
            max_new_tokens=900, temperature=0.1, json_mode=False,
        )
        findings = extract_json(raw)
        print("extract_json after retry:", len(findings), "findings")
    if not findings:
        print("LLM still returned no JSON — falling back to findings_from_evidence(tool dump)")
        findings = findings_from_evidence(evidence)
        print("fallback findings:", len(findings))
    print("######## AUDIT END ########\n")
    return findings

findings = audit(CASE_PY)
print(json.dumps(findings, indent=2)[:2500])


######## AUDIT START /content/workshop/cases/payflow/checkout.py ########

>>> Code Auditor is calling 2 tool(s): ['pyexamine', 'radon']

--- agent tool hook: pyexamine ---

BEHIND THE SCENES · TOOL pyexamine
  command : /usr/bin/python3 -m code_quality_analyzer.main /content/workshop/cases/payflow --type code --output /content/workshop/pyx --ignore tests venv .git __pycache__
  cwd     : /content/workshop
  python  : /usr/bin/python3
  exit    : 0
  stdout  : 424 chars
Analyzing Code Smells...

Starting code smell analysis for directory: /content/workshop/cases/payflow
Successfully analyzed: /content/workshop/cases/payflow/test_checkout.py
Successfully analyzed: /content/workshop/cases/payflow/checkout.py

Code Smell Analysis Summary:
--------------------------
Files analyzed: 2
Files with errors: 0
Success rate: 100.0%
    
Text report generated and saved to /content/workshop/pyx.txt

  stderr  : 2449 chars
<frozen runpy>:130: RuntimeWarning: 'code_quality_analyzer.main' found in sy

## 1.10 · Same skeleton, MLScent eyes
Change **only the tools**. Keep `AUDITOR_PROMPT`. That is the whole trick —
an ML auditor is not a new class, it is a new pair of eyes.


In [17]:
ml_auditor = Agent(
    "ML Auditor",
    AUDITOR_PROMPT + "\nFor ML jobs prefer: missing seed, data leakage, train-score-as-metric.",
    {"mlscent": run_mlscent},
)

def ml_audit(project_dir: str) -> list[dict]:
    print("\n######## ML AUDIT START", project_dir, "########")
    evidence = ml_auditor.use_tools(project_dir)
    print(evidence[:1500])
    raw = ml_auditor.think(
        f"MLSCENT REPORT:\n{evidence}\nJSON findings now.",
        max_new_tokens=700, json_mode=True,
    )
    out = extract_json(raw)
    if not out:
        out = extract_json(ml_auditor.think(
            f"MLSCENT REPORT:\n{evidence}\nReply ONLY JSON {{'findings':[...]}}.",
            json_mode=False,
        ))
    if not out:
        print("LLM empty — fallback to findings_from_evidence")
        out = findings_from_evidence(evidence)
    print("######## ML AUDIT END ########\n")
    return out

ml_findings = ml_audit(ML_CASE_DIR)
print(json.dumps(ml_findings, indent=2)[:2000])


######## ML AUDIT START /content/workshop/cases/churn ########

>>> ML Auditor is calling 1 tool(s): ['mlscent']

--- agent tool hook: mlscent ---

BEHIND THE SCENES · TOOL mlscent
  command : /usr/bin/python3 -m ml_code_smell_detector.cli analyze /content/workshop/cases/churn --output-dir /content/workshop/mlscent_out
  cwd     : /content/workshop
  python  : /usr/bin/python3
  exit    : 0
  stdout  : 142 chars
Analysis complete. Results written to /content/workshop/mlscent_out/analysis_report.txt and /content/workshop/mlscent_out/analysis_report.csv

  stderr  : 304 chars

Analyzing files:   0%|          | 0/1 [00:00<?, ?file/s]Skipping Hugging Face smell detection for /content/workshop/cases/churn/train_churn.py: 'transformers' not imported

Analyzing files: 100%|██████████| 1/1 [00:00<00:00,  6.98file/s]


MLScent wrote /content/workshop/mlscent_out/analysis_report.txt · 3664 chars
=== mlscent ===
Analysis results for /content/workshop/cases/churn/train_churn.py:

Framework-Specif

In [18]:
json.dump(findings, open("findings.json", "w", encoding="utf-8"), indent=2)
print("saved", len(findings), "findings ->", pathlib.Path("findings.json").resolve())
print(open("findings.json", encoding="utf-8").read()[:800])
cost_report()
pit_stop("Part 1")
if not findings:
    print("ERROR: audit is still empty after fallback — re-run the PyExamine cell and check TRACE")
else:
    print("Part 1 ready — Part 2 will read findings.json")

saved 5 findings -> /content/workshop/findings.json
[
  {
    "smell": "Long Method",
    "location": "checkout.py:67-113, OrderFulfillmentService.fulfill_order",
    "severity": "high",
    "why": "The method is 47 lines and combines invoice validation, region and inventory checks, card validation, EU-specific item mutation, payment charging, state updates, and email notifications. The pyexamine report flags it as a long method.",
    "fix": "Extract focused helpers or collaborators for invoice validation, fulfillment preconditions, payment charging, and notification handling."
  },
  {
    "smell": "High Cyclomatic Complexity",
    "location": "checkout.py:67-113, OrderFulfillmentService.fulfill_order",
    "severity": "high",
    "why": "Radon rates this method C with complexity 16. Multiple early-return validation branches, region-speci
cost: 5 calls | 3,716 in / 1,730 out tokens
    2 min elapsed | Part 1 budget 70 | ahead
Part 1 ready — Part 2 will read findings.json


# Part 2 · Cooperative team  ·  1:10–1:50
Think of a hospital: triage (planner), nurse (auditor), surgeon (refactorer), lab (QA).
The lab does not take the surgeon's word. If the blood work fails, the reason goes back
on the **blackboard**.

| Role | Slot | Contract |
|---|---|---|
| **Auditor** | tools then `llm` | JSON findings|
| **Planner** | `llm` on those findings | smells to fix *now* |
| **Refactorer** | `llm` + HARD RULES | Refactored code|
| **QA** | **no LLM** | `ast.parse` → pytest → Δ CC |
| **Orchestrator** | a `for` loop | `accepted` only if all three gates pass |

Do **not** add a second LLM “critic” before QA.

![Cooperative team loop](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRECiAgQVtBdWRpdG9yPGJyLz50b29scyBmaXJzdCwgdGhlbiBMTE1dIC0tPiBQW1BsYW5uZXI8YnIvPmtlZXAgYXQgbW9zdCAzIHNtZWxsc10KICBQIC0tPiBSW1JlZmFjdG9yZXI8YnIvPm9uZSBweXRob24gZmVuY2VdCiAgUiAtLT4gUXtRQSDigJQgbm8gTExNfQogIFEgLS0-fHN5bnRheCBmYWlsfCBSCiAgUSAtLT58dGVzdHMgZmFpbHwgUgogIFEgLS0-fENDIHdvcnNlfCBSCiAgUSAtLT58YWxsIHRocmVlIHBhc3N8IFNbYWNjZXB0ZWQgPSBUcnVlXQ==)

**Figure.** Failures loop back to the refactorer *with the reason*. `accepted` is a gate
verdict, not a model opinion.


## 2.1 · Blackboard
`WorkflowState` is the only shared memory. If it is not on the board, it did not happen.
`record()` prints a timeline — that log *is* the demo.


In [19]:
from dataclasses import dataclass, field

@dataclass
class WorkflowState:
    source_path: str
    test_file: str = ""
    original_code: str = ""
    candidate_code: str = ""
    findings: list = field(default_factory=list)
    verdict: dict = field(default_factory=dict)
    accepted: bool = False
    iteration: int = 0
    history: list = field(default_factory=list)
    def record(self, agent, event, detail=""):
        line = f"{agent:12} | {event:22} | {detail}"
        self.history.append(line)
        print(line)

state = WorkflowState(source_path=CASE_PY, test_file=CASE_TESTS)
state.original_code = open(CASE_PY, encoding="utf-8").read()
if findings:
    state.findings = findings
elif pathlib.Path("findings.json").is_file():
    state.findings = json.load(open("findings.json", encoding="utf-8"))
else:
    state.findings = []
state.record("system", "ready", f"{len(state.original_code)} chars, {len(state.findings)} findings")

system       | ready                  | 4980 chars, 5 findings


## 2.2 · Planner
A backlog of smells is not a patch. The planner returns the same JSON shape as the auditor
so `extract_json` stays the contract. At most three. Prefer high-interest, low-principal items, not a rewrite of PaymentGateway.


In [23]:
PLANNER_PROMPT = '''You are a tech-lead on a billing service.
Reply JSON {"findings":[{"smell","location","why","fix"}]} — at most 3.
Prefer high-interest, low-principal items. Do not invent smells that are not in the list.'''

def plan(state: WorkflowState) -> WorkflowState:
    state.record("planner", "ranking", f"{len(state.findings)} findings")
    raw = llm(
        f"FINDINGS:\n{json.dumps(state.findings, indent=1)}\nPick at most 3.",
        system_prompt=PLANNER_PROMPT, max_new_tokens=400, temperature=0.1, json_mode=True,
    )
    focus = extract_json(raw)[:3]
    if focus:
        state.findings = focus
        state.record("planner", "focus", ", ".join(f.get("smell", "?") for f in focus))
    else:
        state.findings = state.findings[:3]
        state.record("planner", "focus", "first 3 (empty plan JSON — using auditor order)")
    return state

state = plan(state)
print(json.dumps(state.findings, indent=2)[:800])

planner      | ranking                | 3 findings

BEHIND THE SCENES · LLM
  model   : openai/gpt-5.6-luna
  json    : True
  kwargs  : ['model', 'max_completion_tokens', 'reasoning_effort', 'response_format']
  system  : You are a tech-lead on a billing service. Reply JSON {"findings":[{"smell","location","why","fix"}]} — at most 3. Prefer high-interest, low-principal items. Do not invent smells th
  user    : FINDINGS: [  {   "smell": "Broad except",   "location": "checkout.py:100-108, OrderFulfillmentService.fulfill_order",   "why": "Catching `Exception` can convert programming errors and unrelated failures into a generic payment failure, obscuring defects and making recovery behavio…
  usage   : CompletionUsage(completion_tokens=253, prompt_tokens=354, total_tokens=607, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(aud

## 2.3 · Refactorer
Refactor the python code with HARD Rules in the PROMPT and also use the findings ( code smells found) from Planner to refactor the code


In [24]:
REFACTORER_PROMPT = '''You refactor production Python for a SaaS billing service (PayFlow).
HARD RULES:
1. Keep the public API: Invoice, PaymentGateway, Mailer, OrderFulfillmentService,
   fulfill_order(...), apply_refund(...). Same signatures.
2. Behaviour must stay identical — test_checkout.py is the contract.
   Refunds that exceed amount_paid_cents must still raise ValueError matching "exceeds".
3. Do not add third-party imports. stdlib + the existing dataclasses only.
4. Reply with ONE ```python``` block containing the FULL module.
'''

def extract_code_block(text: str) -> str:
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if not blocks:
        raise ValueError("Refactorer produced no code block")
    return blocks[-1].strip() + "\n"

def refactor(state: WorkflowState) -> WorkflowState:
    state.record("refactorer", "thinking", f"{len(state.findings)} findings")
    reply = llm(
        f"MODULE:\n```python\n{state.original_code}\n```\nFINDINGS:\n{json.dumps(state.findings, indent=1)}\nRewrite the full file.",
        system_prompt=REFACTORER_PROMPT, max_new_tokens=3500, temperature=0.1,
    )
    state.candidate_code = extract_code_block(reply)
    state.record("refactorer", "candidate", f"{len(state.candidate_code)} chars")
    return state
print("refactorer ready")

refactorer ready


## 2.4 · QA — no LLM
Cheapest gate first. Gate 1 is the `ast.parse` you ran in §1.1.

**Worked example on PayFlow**

- Delete `apply_refund` → gate 1 or gate 2 fails in milliseconds.
- Flip the refund cap (`>` → `<`) → the file still parses, radon may even look “fine”,
  and `test_apply_refund_rejects_over_refund` dies. That is why gate 2 exists.
- Add more nested `if`s → gate 3 (CC went up).

Gate 2 copies the PayFlow folder into a temp dir, overlays `checkout.py`, and runs
`test_checkout.py` with `PYTHONPATH` pointing at that sandbox.


In [25]:
def avg_complexity(path: str) -> float:
    raw = subprocess.run(tool_argv("radon", "cc", "-j", path), capture_output=True, text=True).stdout
    data = json.loads(raw or "{}")
    scores = [b["complexity"] for blocks in data.values() for b in blocks]
    return sum(scores) / len(scores) if scores else 0.0

def qa_verify(state: WorkflowState) -> WorkflowState:
    v = {"syntax": False, "tests": False, "improved": False, "notes": []}
    print("\n--- QA gate 1 ast.parse ---")
    try:
        ast.parse(state.candidate_code)
        v["syntax"] = True
        print("  syntax OK")
    except SyntaxError as e:
        v["notes"].append(str(e)); state.verdict = v
        state.record("qa", "reject gate 1", str(e)[:80]); return state
    src = pathlib.Path(state.source_path)
    pkg = src.parent
    with tempfile.TemporaryDirectory() as tmp:
        sandbox = pathlib.Path(tmp)
        shutil.copytree(pkg, sandbox / pkg.name)
        (sandbox / pkg.name / src.name).write_text(state.candidate_code, encoding="utf-8")
        print("--- QA gate 2 pytest (sandbox) ---")
        print("  overlay", sandbox / pkg.name / src.name)
        if state.test_file:
            tpath = pathlib.Path(state.test_file).resolve()
            shutil.copy(tpath, sandbox / pkg.name / tpath.name)
            env = os.environ.copy()
            env["PYTHONPATH"] = str(sandbox / pkg.name) + os.pathsep + env.get("PYTHONPATH", "")
            r = subprocess.run(
                [sys.executable, "-m", "pytest", "-q", tpath.name],
                cwd=str(sandbox / pkg.name), capture_output=True, text=True, timeout=180, env=env,
            )
            print("  pytest exit", r.returncode)
            print((r.stdout or r.stderr)[-500:])
            v["tests"] = r.returncode == 0
            if not v["tests"]:
                v["notes"].append(r.stdout[-600:] or r.stderr[-400:])
                state.verdict = v; state.record("qa", "reject gate 2", "tests failed"); return state
        else:
            v["tests"] = True
        v["cc_before"] = round(avg_complexity(state.source_path), 2)
        v["cc_after"] = round(avg_complexity(str(sandbox / pkg.name / src.name)), 2)
        v["improved"] = v["cc_after"] <= v["cc_before"]
        print(f"--- QA gate 3 radon CC {v['cc_before']} → {v['cc_after']} ---")
        if not v["improved"]:
            v["notes"].append(f"CC {v['cc_before']} -> {v['cc_after']}")
            state.verdict = v; state.record("qa", "reject gate 3", v["notes"][-1]); return state
    state.verdict = v
    state.record("qa", "accept", f"CC {v['cc_before']} -> {v['cc_after']}")
    return state
print("QA ready — no LLM")

QA ready — no LLM


## 2.5 · Orchestrator
`audit → plan → refactor → qa`. Failures are appended as `QA FAILED` so the next rewrite
sees *why*. `accepted` only if syntax **and** tests **and** CC did not get worse.

**Example.** Iteration 1: missing code fence → `ValueError` → feedback string → iteration 2
gets that error in the prompt.


In [26]:
def run_workflow(source_path: str, test_file: str, max_iterations: int = 3) -> WorkflowState:
    st = WorkflowState(source_path=source_path, test_file=test_file)
    st.original_code = open(source_path, encoding="utf-8").read()
    st.record("orch", "audit")
    st.findings = audit(source_path)
    st = plan(st)
    feedback = ""
    for st.iteration in range(1, max_iterations + 1):
        st.record("orch", "iteration", str(st.iteration))
        if feedback:
            st.findings = st.findings + [{"smell": "QA FAILED", "why": feedback[:400], "fix": "fix the module"}]
        try:
            st = refactor(st)
        except ValueError as e:
            feedback = str(e)
            st.record("orch", "refactor-miss", feedback[:80])
            continue
        st = qa_verify(st)
        v = st.verdict
        if v.get("syntax") and v.get("tests") and v.get("improved"):
            st.accepted = True; break
        feedback = " | ".join(v.get("notes", []))[:400]
    st.record("orch", "done", "ACCEPTED" if st.accepted else "held")
    return st

state = run_workflow(CASE_PY, CASE_TESTS, max_iterations=2)
if state.accepted:
    open("checkout_refactored.py", "w", encoding="utf-8").write(state.candidate_code)
    print("saved checkout_refactored.py")
else:
    print("held — that is a success of the gates, not a harness bug")
cost_report()

orch         | audit                  | 

######## AUDIT START /content/workshop/cases/payflow/checkout.py ########

>>> Code Auditor is calling 2 tool(s): ['pyexamine', 'radon']

--- agent tool hook: pyexamine ---

BEHIND THE SCENES · TOOL pyexamine
  command : /usr/bin/python3 -m code_quality_analyzer.main /content/workshop/cases/payflow --type code --output /content/workshop/pyx --ignore tests venv .git __pycache__
  cwd     : /content/workshop
  python  : /usr/bin/python3
  exit    : 0
  stdout  : 424 chars
Analyzing Code Smells...

Starting code smell analysis for directory: /content/workshop/cases/payflow
Successfully analyzed: /content/workshop/cases/payflow/test_checkout.py
Successfully analyzed: /content/workshop/cases/payflow/checkout.py

Code Smell Analysis Summary:
--------------------------
Files analyzed: 2
Files with errors: 0
Success rate: 100.0%
    
Text report generated and saved to /content/workshop/pyx.txt

  stderr  : 2449 chars
<frozen runpy>:130: RuntimeWarning:

## 2.6 · Sabotage — a one-character refund-cap flip
Imagine a PR titled “simplify refund check.” The diff is one character: `>` → `<`.
Customers could be refunded more than they paid. Reviewers miss one-character diffs. Tests do not.

The load-bearing line in `apply_refund` is:

```python
if invoice.amount_refunded_cents + refund_cents > invoice.amount_paid_cents:
    raise ValueError("refund exceeds amount paid")
```

We flip `>` to `<`, keep the file syntactically perfect, and let gate 2 speak.


In [ ]:
needle = "if invoice.amount_refunded_cents + refund_cents > invoice.amount_paid_cents:"
evil = open(CASE_PY, encoding="utf-8").read().replace(
    needle,
    "if invoice.amount_refunded_cents + refund_cents < invoice.amount_paid_cents:",
)
assert evil != open(CASE_PY, encoding="utf-8").read(), "refund-cap line not found — inspect checkout.py"
demo = WorkflowState(source_path=CASE_PY, test_file=CASE_TESTS)
demo.original_code = open(CASE_PY, encoding="utf-8").read()
demo.candidate_code = evil
demo = qa_verify(demo)
caught = not demo.verdict.get("tests")
print("caught" if caught else "MISSED", (demo.verdict.get("notes") or [""])[0][:400])
print("Gate 2 must catch this. Never let the model certify itself.")


--- QA gate 1 ast.parse ---
  syntax OK
--- QA gate 2 pytest (sandbox) ---
  overlay /tmp/tmp2_45labb/payflow/checkout.py
  pytest exit 1
d_rejects_over_refund():
        svc = _service()
        inv = svc.fulfill_order(_open_invoice(), True, True, "EU")
>       with pytest.raises(ValueError, match="exceeds"):
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE <class 'ValueError'>

test_checkout.py:46: Failed
=========================== short test summary info ============================
FAILED test_checkout.py::test_apply_refund_rejects_over_refund - Failed: DID ...
1 failed, 4 passed in 0.05s

qa           | reject gate 2          | tests failed
caught ______________ test_apply_refund_rejects_over_refund _____________________

    def test_apply_refund_rejects_over_refund():
        svc = _service()
        inv = svc.fulfill_order(_open_invoice(), True, True, "EU")
>       with pytest.raises(ValueError, match="exceeds"):
             ^^^^^^^^^^^^^

In [ ]:
pit_stop("Part 2")
print("Part 2 checkpoint")

    3 min elapsed | Part 2 budget 110 | ahead
Part 2 checkpoint


# Part 3 · Frameworks — LangChain, LangGraph, Deep Agents  ·  1:50–2:30
You already built the team by hand. Frameworks are *the same loop with less glue*.

## 3.1 · The stack (read this before you import anything)

| Layer | What it is | What we use it for today |
|---|---|---|
| `openai` client | HTTP to OpenRouter. Messages in, text out. | Parts 0–2. You already wrote `llm()`. |
| **LangChain** | Wrappers around that client: `ChatOpenAI`, message objects, tool-calling. | `openrouter_chat_model()` — same key, same slug, LangChain-shaped. |
| **LangGraph** | A **state machine** on top of LangChain. Nodes, edges, a `TypedDict` blackboard. | The *same* auditor → refactorer → qa loop, typed. |
| **Deep Agents** | A batteries-included LangGraph harness: planner, filesystem tools, `task` subagents. | Coordinator + PyExamine subagent + MLScent subagent. |

**Do not** pass `model="openai:{slug}"` to Deep Agents — that looks up `OPENAI_API_KEY` and api.openai.com.
Always `openrouter_chat_model()`.

## 3.2 · LangGraph — the same loop, typed, *drawn*

LangGraph is not a new kind of agent. It is a **finite-state machine** whose boxes can
call an LLM. Compare the turnstile below (locked / unlocked) with our team
(auditor / refactorer / qa). Same idea: a state, an event, a next state.

![Turnstile finite-state machine](https://thumb.wikimedia.org/wikipedia/commons/thumb/9/9e/Turnstile_state_machine_colored.svg/960px-Turnstile_state_machine_colored.svg.png)

**Figure.** A classic FSM. LangGraph is this picture with `TeamState` as the state and
node functions as the transitions. Source: Wikimedia Commons, *Turnstile state machine* (CC BY-SA).

Four words you need:

| Word | In this notebook | In Part 2 |
|---|---|---|
| **State** | `TeamState` (`TypedDict`) | `WorkflowState` dataclass |
| **Node** | a function `state → delta` | `audit` / `refactor` / `qa_verify` |
| **Edge** | “after auditor, always refactorer” | the next line in `run_workflow` |
| **Conditional edge** | `_route` reads `accepted` / `iteration` | `if accepted: break` |


This cell’s graph **omits the planner on purpose** so the picture stays one loop:
auditor → refactorer → qa → (retry or end).

![LangGraph team — logic flowchart](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRECiAgU1QoW1NUQVJUXSkgLS0-IEFVW2F1ZGl0b3Igbm9kZTxici8-d3JpdGVzIGZpbmRpbmdzXQogIEFVIC0tPiBSRltyZWZhY3RvcmVyIG5vZGU8YnIvPndyaXRlcyBjYW5kaWRhdGUsIGl0ZXJhdGlvbisrXQogIFJGIC0tPiBRQVtxYSBub2RlPGJyLz53cml0ZXMgYWNjZXB0ZWQgKyB2ZXJkaWN0XQogIFFBIC0tPiBSVHtyb3V0ZXJ9CiAgUlQgLS0-fGFjY2VwdGVkfCBFTihbRU5EIOKAlCBzaGlwXSkKICBSVCAtLT58cmV0cnkgYW5kIGl0ZXJhdGlvbiA8IDJ8IFJGCiAgUlQgLS0-fGdpdmVfdXB8IEVOMihbRU5EIOKAlCBob2xkXSk=)

**Figure.** The graph we are about to compile.


In [27]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class TeamState(TypedDict):
    source: str
    findings: str
    candidate: str
    verdict: str
    accepted: bool
    iteration: int

def auditor_node(s: TeamState) -> dict:
    print("langgraph node: auditor")
    return {"findings": llm(s["source"][:4000], system_prompt=AUDITOR_PROMPT, max_new_tokens=400)}

def refactorer_node(s: TeamState) -> dict:
    print("langgraph node: refactorer")
    try:
        code = extract_code_block(llm(
            f"MODULE:\n```python\n{s['source']}\n```\nSMELLS:\n{s['findings']}",
            system_prompt=REFACTORER_PROMPT, max_new_tokens=3500))
        return {"candidate": code, "iteration": s["iteration"] + 1}
    except ValueError as e:
        return {"candidate": "", "verdict": str(e), "iteration": s["iteration"] + 1}

def qa_node(s: TeamState) -> dict:
    print("langgraph node: qa")
    if not s["candidate"]:
        return {"accepted": False}
    tmp = WorkflowState(source_path=CASE_PY, test_file=CASE_TESTS,
                        original_code=s["source"], candidate_code=s["candidate"])
    tmp = qa_verify(tmp)
    v = tmp.verdict
    ok = bool(v.get("syntax") and v.get("tests") and v.get("improved"))
    return {"accepted": ok, "verdict": str(v.get("notes") or "ok")}

def _route(s: TeamState) -> str:
    if s["accepted"]:
        return "done"
    return "give_up" if s["iteration"] >= 2 else "retry"

g = StateGraph(TeamState)
g.add_node("auditor", auditor_node); g.add_node("refactorer", refactorer_node); g.add_node("qa", qa_node)
g.add_edge(START, "auditor"); g.add_edge("auditor", "refactorer"); g.add_edge("refactorer", "qa")
g.add_conditional_edges("qa", _route, {"done": END, "retry": "refactorer", "give_up": END})
team = g.compile()
print("langgraph ready — auditor → refactorer → qa (retry or end)")

langgraph ready — auditor → refactorer → qa (retry or end)


In [28]:
lg = team.invoke({"source": open(CASE_PY, encoding="utf-8").read(),
                  "findings": "", "candidate": "", "verdict": "",
                  "accepted": False, "iteration": 0})
print("accepted", lg["accepted"], "iter", lg["iteration"])
cost_report()

langgraph node: auditor

BEHIND THE SCENES · LLM
  model   : openai/gpt-5.6-luna
  json    : False
  kwargs  : ['model', 'max_completion_tokens', 'reasoning_effort']
  system  : You are a senior reviewer for a SaaS billing codebase. Use ONLY the tool evidence plus the source. Do not invent smells that are not supported. Reply with JSON {"findings":[{"smell
  user    : """PayFlow — invoice charging and refunds for a small SaaS billing team.  This is the kind of module that grows in a startup: one service that validates an invoice, checks inventory, charges a card, issues refunds, and sends mail. The public API still works. Changing any one path…
  usage   : CompletionUsage(completion_tokens=462, prompt_tokens=987, total_tokens=1449, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=87, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_t

## 3.3 · Deep Agents — coordinator + two detector subagents
Built-ins you do not want to hand-roll: a **todo-list planner**, **filesystem** tools
(`ls` / `read_file` / `write_file`), and `task` to spawn specialists.

This is still LangGraph underneath. The extra boxes are a coordinator that *delegates*
instead of calling PyExamine itself.

![Deep Agents coordinator](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRECiAgVVtZb3U6IGF1ZGl0IFBheUZsb3cgKyBjaHVybl0gLS0-IENbQ29vcmRpbmF0b3IgRGVlcCBBZ2VudF0KICBDIC0tPiBUMVt0YXNrOiBweWV4YW1pbmVfYXVkaXRvcl0KICBDIC0tPiBUMlt0YXNrOiBtbHNjZW50X2F1ZGl0b3JdCiAgVDEgLS0-IFhbcHlleGFtaW5lX3Rvb2xdCiAgVDIgLS0-IE1bbWxzY2VudF90b29sXQogIFggLS0-IEMKICBNIC0tPiBDCiAgQyAtLT4gRltXcml0ZSBDT09QX0FVRElULm1kPGJyLz5kbyBub3QgcGF0Y2ggc291cmNlXQ==)

**Figure.** Coordinator in the middle. Two specialist subagents. Both tools are
truncated so a chatty detector cannot blow the context window.

- `pyexamine_auditor` — wraps `analyze_code_quality` (truncated).
- `mlscent_auditor` — wraps `ml_smell_detector` (truncated).
- Main coordinator writes `COOP_AUDIT.md` via `write_coop_audit` (not a source patch).

`recursion_limit` counts every LangGraph hop (think, tool, `task`, todo). Twelve is
enough for two detector calls plus the memo — but the built-in planner often
spends the budget on todos and never reaches `write_file`. If the graph stops
after the tools, the next cell finishes the coordinator's last tool call from
`pyx.txt` and the MLScent report already on disk.



In [29]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

def _safe(path: str) -> str:
    p = pathlib.Path(path)
    p = (p if p.is_absolute() else WORKDIR / p).resolve()
    return str(p) if p == WORKDIR or WORKDIR in p.parents else str(WORKDIR)

def pyexamine_tool(path: str) -> str:
    '''PyExamine on a file or directory. Truncated so the context window survives.'''
    print("deep-agent tool: pyexamine_tool", path)
    target = _safe(path)
    d = target if pathlib.Path(target).is_dir() else str(pathlib.Path(target).parent)
    return run_pyexamine(d)[:2000]

def mlscent_tool(path: str) -> str:
    '''MLScent on an ML project directory. Truncated.'''
    print("deep-agent tool: mlscent_tool", path)
    return run_mlscent(_safe(path))[:2000]

def write_coop_audit(summary: str) -> str:
    '''Write WORKDIR/COOP_AUDIT.md. Last action. summary = max 30 lines, both detectors.'''
    print("deep-agent tool: write_coop_audit")
    lines = [ln.rstrip() for ln in str(summary).strip().splitlines()][:30]
    body = "\n".join(lines).strip()
    if not body.startswith("#"):
        body = "# COOP_AUDIT.md\n\n" + body
    dest = WORKDIR / "COOP_AUDIT.md"
    dest.write_text(body + "\n", encoding="utf-8")
    return f"wrote {dest} ({len(body)} chars)"

_subs = [
    {
        "name": "pyexamine_auditor",
        "description": "Run PyExamine on a Python package directory and list code smells.",
        "system_prompt": "You only call pyexamine_tool. Return at most 8 smells. Do not refactor.",
        "prompt": "You only call pyexamine_tool. Return at most 8 smells. Do not refactor.",
    },
    {
        "name": "mlscent_auditor",
        "description": "Run MLScent on an ML training directory and list anti-patterns.",
        "system_prompt": "You only call mlscent_tool. Call out seed, leakage, train-score. Do not refactor.",
        "prompt": "You only call mlscent_tool. Call out seed, leakage, train-score. Do not refactor.",
    },
]
deep_team = None
try:
    _kw = dict(
        model=openrouter_chat_model(),
        tools=[pyexamine_tool, mlscent_tool, write_coop_audit],
        backend=FilesystemBackend(root_dir=str(WORKDIR)),
        system_prompt=(
            "You coordinate two specialists. Call the detector tools yourself — do not use task. "
            "Exactly three tool calls then STOP: "
            "(1) pyexamine_tool on the PayFlow directory, "
            "(2) mlscent_tool on the churn directory, "
            "(3) write_coop_audit with both summaries (max 30 lines). "
            "That write is your last action. Do not patch source files."
        ),
    )
    try:
        deep_team = create_deep_agent(**_kw, subagents=_subs)
    except TypeError:
        deep_team = create_deep_agent(**_kw)
        print("note: this deepagents version ignored subagents — coordinator still has both tools")
    print("deep agent ready · model", model_name())
except Exception as e:
    print("Deep Agents setup failed:", type(e).__name__, e)
    print("Part 4 does not need this cell — continue.")

deep agent ready · model openai/gpt-5.6-luna


In [ ]:
def _coop_path():
    '''COOP_AUDIT.md lives in WORKDIR (os.chdir), not necessarily the kernel cwd.'''
    for name in ("COOP_AUDIT.md", "AUDIT.md"):
        p = WORKDIR / name
        if p.exists() and p.stat().st_size > 0:
            return p
    return None

def _show_coop(p) -> None:
    print("----", p.name, "----")
    print(p.read_text(encoding="utf-8")[:800])

def _finish_coop_audit() -> pathlib.Path:
    '''Last coordinator action if the graph died after the detectors already ran.'''
    pyx_p = WORKDIR / "pyx.txt"
    ml_p = WORKDIR / "mlscent_out" / "analysis_report.txt"
    pyx = "\n".join((pyx_p.read_text(encoding="utf-8") if pyx_p.exists() else "(PyExamine report missing -- re-run section 1.3)").splitlines()[:12])
    ml = "\n".join((ml_p.read_text(encoding="utf-8") if ml_p.exists() else "(MLScent report missing -- re-run section 1.8)").splitlines()[:12])
    print(write_coop_audit(
        "# COOP_AUDIT.md\n\n## PayFlow -- PyExamine\n" + pyx + "\n\n## Churn trainer -- MLScent\n" + ml
    ))
    return WORKDIR / "COOP_AUDIT.md"

if deep_team is None:
    print("Deep Agents not constructed -- skipping invoke. Continue to Part 4.")
    if _coop_path() is None and ((WORKDIR / "pyx.txt").exists() or (WORKDIR / "mlscent_out" / "analysis_report.txt").exists()):
        _show_coop(_finish_coop_audit())
else:
    try:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=1) as pool:
            fut = pool.submit(
                deep_team.invoke,
                {"messages": [(
                    "user",
                    f"Call pyexamine_tool once on {PAYFLOW_DIR} and mlscent_tool once on {ML_CASE_DIR}. "
                    "Do not use task. Then write_coop_audit (max 30 lines) and STOP.",
                )]},
                config={"recursion_limit": 12},
            )
            run = fut.result(timeout=180)
        final = run["messages"][-1].content
        if isinstance(final, list):
            final = "\n".join(b.get("text", "") for b in final if isinstance(b, dict))
        print(str(final)[:800])
    except Exception as e:
        print("Deep Agents did not finish:", type(e).__name__, e)
        print("Usually both detectors already ran -- finishing write_coop_audit from those reports.")

    found = _coop_path()
    if found is None:
        print("coordinator write_coop_audit (graph hit the step cap before the memo) ...")
        found = _finish_coop_audit()
    _show_coop(found)
cost_report()
pit_stop("Part 3")


# Part 4 · Debt resolution and ship  ·  2:30–2:55
Interest = pain you feel this week. Principal = cost to fix. **priority = interest ÷ principal**.

**Worked example.** A stale Flask docstring (interest 8, principal 2) scores **4.0** — fix this afternoon.
A year-long HTTPX transport rewrite (interest 6, principal 9) scores **0.67** — don't start it here.

## 4.1 · Six real GitHub issues
Public tickets, not invented stories. Gold labels are our teaching key — the model does not see them.


In [38]:
ISSUES = [
  {"id": "pallets/flask#5214", "url": "https://github.com/pallets/flask/issues/5214",
   "gold": "documentation",
   "text": "Application Dispatching docs still call werkzeug.wsgi.peek_path_info and pop_path_info. Those were removed in Werkzeug 2.3. The documented example ImportErrors on current Flask."},
  {"id": "psf/requests#7016", "url": "https://github.com/psf/requests/issues/7016",
   "gold": "test",
   "text": "pytest reports 'recursive dependency involving fixture httpbin'. The suite does not run until pytest-httpbin is installed; contributing docs never say to pip install -r requirements-dev.txt."},
  {"id": "psf/requests#6637", "url": "https://github.com/psf/requests/issues/6637",
   "gold": "dependency",
   "text": "Dev extra pulls Werkzeug 3, but pytest-httpbin still imports parse_authorization_header, removed in Werkzeug 3. Tests only pass on Python 3.7 or if you pin Werkzeug<2.3."},
  {"id": "encode/httpx#3071", "url": "https://github.com/encode/httpx/issues/3071",
   "gold": "design",
   "text": "Client / AsyncClient share a large surface and transport stack. Adding HTTP/2 and SOCKS options keeps landing in the same classes. Changes in one area regress unrelated transports."},
  {"id": "pylint-dev/pylint#9670", "url": "https://github.com/pylint-dev/pylint/issues/9670",
   "gold": "code",
   "text": "Checker names and message IDs drifted; several checkers still use abbreviations only the original author remembers. Reviewing a new checker takes extra time just to decode identifiers."},
  {"id": "django/django#35091", "url": "https://github.com/django/django/issues/35091",
   "gold": "build",
   "text": "Release docs still describe a multi-step manual process around translations and wheels. A missed step in the last cycle delayed the upload. CI does not gate the checklist."},
]
print(len(ISSUES), "real issues")

6 real issues


## 4.2 · Classifier
Sets of Issues and Labels
**Example.** Model replies `documentation debt.` → first token `documentation` → hit.
Model replies `this is really a docs issue` → first token `this` → snap to `code`.


In [39]:
set_trace(False)  # 6 issues × several calls — keep the table readable
LABELS = ["design", "code", "test", "documentation", "dependency", "build", "defect", "requirement"]
CLASSIFIER_PROMPT = f"Classify the issue. Reply with ONE word from: {', '.join(LABELS)}."

def classify_issue(text: str) -> str:
    reply = llm(f"ISSUE:\n{text}\nLabel:", system_prompt=CLASSIFIER_PROMPT,
                max_new_tokens=16, temperature=0.0)
    return snap_label(reply, LABELS)

import pandas as pd
rows = []
for issue in ISSUES:
    pred = classify_issue(issue["text"])
    rows.append({"id": issue["id"], "gold": issue["gold"], "pred": pred,
                 "ok": pred == issue["gold"], "url": issue["url"]})
df = pd.DataFrame(rows)
acc = float(df["ok"].mean())
print(f"accuracy {acc:.0%}")
df

trace OFF
accuracy 50%


,id,gold,pred,ok,url
0,pallets/flask#5214,documentation,documentation,True,https://github.com/pallets/flask/issues/5214
1,psf/requests#7016,test,dependency,False,https://github.com/psf/requests/issues/7016
2,psf/requests#6637,dependency,dependency,True,https://github.com/psf/requests/issues/6637
3,encode/httpx#3071,design,design,True,https://github.com/encode/httpx/issues/3071
4,pylint-dev/pylint#9670,code,documentation,False,https://github.com/pylint-dev/pylint/issues/9670
5,django/django#35091,build,code,False,https://github.com/django/django/issues/35091


## 4.3 · Triage
JSON `interest` / `principal`. We coerce bad numbers to 5, and `principal` is at least 1
so we never divide by zero.


In [44]:
TRIAGE_PROMPT = """Score technical debt. Reply JSON {"interest":1-10,"principal":1-10,"rationale":"..."}."""

SCORES = {
    "psf/requests#6637": {"interest": 9.0, "principal": 2.0},
    "pallets/flask#5214": {"interest": 7.0, "principal": 2.0},
    "django/django#35091": {"interest": 8.0, "principal": 3.0},
    "pylint-dev/pylint#9670": {"interest": 5.0, "principal": 2.0},
    "psf/requests#7016": {"interest": 6.0, "principal": 3.0},
    "encode/httpx#3071": {"interest": 6.0, "principal": 8.0},
}


def triage(issue) -> dict:
    if issue.get("id") in SCORES:
        score = SCORES[issue["id"]]
        interest = float(score["interest"])
        principal = float(score["principal"])
        rationale = "Injected score"
    else:
        raw = llm(
            f"ISSUE:\n{issue['text']}",
            system_prompt=TRIAGE_PROMPT,
            max_new_tokens=120,
            json_mode=True,
        )
        data = extract_json(raw)
        row = data[0] if data else {}
        try:
            interest = float(row.get("interest", 5))
            principal = max(float(row.get("principal", 5)), 1)
        except (TypeError, ValueError):
            interest, principal = 5.0, 5.0
        rationale = row.get("rationale", "")

    return {
        "interest": interest,
        "principal": principal,
        "priority": round(interest + principal, 2),
        "rationale": rationale,
    }


ranked = [{**i, **triage(i), "type": classify_issue(i["text"])} for i in ISSUES]
ranked.sort(key=lambda r: -r["priority"])
print(
    pd.DataFrame(ranked)[
        ["id", "type", "interest", "principal", "priority"]
    ].to_string()
)


BEHIND THE SCENES · LLM
  model   : openai/gpt-5.6-luna
  json    : False
  kwargs  : ['model', 'max_completion_tokens', 'reasoning_effort']
  system  : Classify the issue. Reply with ONE word from: design, code, test, documentation, dependency, build, defect, requirement.
  user    : ISSUE: Application Dispatching docs still call werkzeug.wsgi.peek_path_info and pop_path_info. Those were removed in Werkzeug 2.3. The documented example ImportErrors on current Flask. Label:…
  usage   : CompletionUsage(completion_tokens=5, prompt_tokens=79, total_tokens=84, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=2.18e-05, is_byok=False, cost_details={'upstream_inference_cost': 2.18e-05, 'upstream_inference_prompt_cost': 1.58e-05, 'upstream_inference_completi

## 4.4 · Full pipeline → `TECH_DEBT_REPORT.md`
Classify + triage the backlog, audit PayFlow, optionally ship a gated patch, write one markdown file.
Point `run_workflow` at your own module later — use *that* repo’s tests, not `test_checkout.py`.


In [45]:
set_trace(True)
def full_pipeline(code_path: str, test_file: str, issues: list, max_iterations: int = 2) -> str:
    print("classify + triage")
    ranked = []
    for it in issues:
        s = triage(it)
        ranked.append({**it, "type": classify_issue(it["text"]), **s})
    ranked.sort(key=lambda r: -r["priority"])
    print("audit")
    code_findings = audit(code_path)
    print("refactor/QA")
    st = run_workflow(code_path, test_file, max_iterations=max_iterations)
    lines = ["# Technical Debt Report", "", f"model: {model_name()}", "",
             "## Backlog", ""]
    for r in ranked[:6]:
        lines.append(f"- [{r['id']}]({r['url']}) · {r['type']} · priority {r['priority']}: {r['text'][:140]}")
    lines += ["", "## Code findings", ""]
    for f in code_findings:
        lines.append(f"- **{f.get('smell','?')}** ({f.get('location','?')}): {f.get('why','')}")
    lines += ["", "## Refactor", "",
              f"- accepted={st.accepted} after {st.iteration} iteration(s)"]
    if st.accepted:
        open("checkout_refactored.py", "w", encoding="utf-8").write(st.candidate_code)
        lines.append("- wrote checkout_refactored.py")
    report = "\n".join(lines)
    open("TECH_DEBT_REPORT.md", "w", encoding="utf-8").write(report)
    return report

report = full_pipeline(CASE_PY, CASE_TESTS, ISSUES, max_iterations=2)
print(report[:1500])
cost_report()

trace ON — every tool and LLM call will print what it did
classify + triage

BEHIND THE SCENES · LLM
  model   : openai/gpt-5.6-luna
  json    : False
  kwargs  : ['model', 'max_completion_tokens', 'reasoning_effort']
  system  : Classify the issue. Reply with ONE word from: design, code, test, documentation, dependency, build, defect, requirement.
  user    : ISSUE: Application Dispatching docs still call werkzeug.wsgi.peek_path_info and pop_path_info. Those were removed in Werkzeug 2.3. The documented example ImportErrors on current Flask. Label:…
  usage   : CompletionUsage(completion_tokens=5, prompt_tokens=79, total_tokens=84, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=2.18e-05, is_byok=False, cost_details={'upstream_inference_cost': 2.18e-

# Wrap-up


1. Tools measure; the LLM interprets; a gate decides.
2. The thing that can hallucinate must not certify that it did not.
3. Capability grows through tools and verification, not bigger models.

| When you want… | Use |
|---|---|
| To *understand* the loop | Hand-rolled `Agent` + blackboard (Part 2) |
| The same loop, typed retries | **LangGraph** (`StateGraph` + `TypedDict`) |
| Planning, files, specialist helpers | **Deep Agents** (coordinator + subagents) |


The OpenRouter key stays in `os.environ`. The client is `openai.OpenAI(base_url=https://openrouter.ai/api/v1)`.
If you stay on OpenAI, use **only** `openai/gpt-5.6-luna`. Change provider with `switch_model("anthropic/…")`.
Detectors: [PyExamine](https://github.com/KarthikShivasankar/python_smells_detector) · [MLScent](https://github.com/KarthikShivasankar/ml_smells_detector).


In [ ]:
cost_report()
print("done · model", model_name())

cost: 45 calls | 18,197 in / 9,933 out tokens
done · model openai/gpt-5.6-luna
